# Python modules

## Teaching notebook

This notebook explains what modules are, why they are useful, how to create them, and how Python finds and imports them.

**Suggested duration:** 45–60 minutes

### Learning outcomes

By the end of the session, students should be able to:

1. Explain what a Python module is.
2. Import and use standard-library modules.
3. Compare common import styles.
4. Create and import a custom `.py` module.
5. Explain the purpose of `if __name__ == "__main__"`.
6. Describe how Python searches for modules.
7. Distinguish modules, packages, and third-party distributions.

## 1. What is a module?

A **module** is a Python file containing reusable code. It can contain functions, classes, constants, variables, imports, and executable statements.

Modules help us:

- break a large program into manageable components;
- reuse tested code in several programs;
- give related code a meaningful namespace;
- make testing and maintenance easier;
- collaborate without everyone editing one enormous file.

> **Teaching analogy:** A module is like a labelled drawer in a workshop. The drawer groups related tools and its label helps us say exactly where a tool came from.

## 2. Using a standard-library module

Python includes a large standard library. The `math` module supplies mathematical functions and constants.

> **Prediction prompt:** Before running the cell, ask students why Python uses `math.sqrt` instead of simply `sqrt`.

In [ ]:
import math

radius = 2.0
area = math.pi * radius**2
root = math.sqrt(25)

print(f"Circle area: {area:.3f}")
print(f"Square root: {root}")

The module name creates a **namespace**. In `math.sqrt`, `math` identifies the namespace and `sqrt` identifies a function inside it. This makes the origin of a name clear and reduces naming conflicts.

## 3. Common import styles

### Import the complete module

```python
import math
answer = math.sqrt(81)
```

This style is explicit and usually easy to read.

### Import selected names

```python
from math import sqrt, pi
answer = sqrt(81)
```

This is convenient, but the module name is no longer visible at the point of use.

### Use an alias

```python
import statistics as stats
average = stats.mean([10, 12, 15])
```

Aliases are useful when they are conventional or make a long name easier to use.

In [ ]:
import statistics as stats
from math import pi, sqrt

measurements = [12.1, 12.5, 11.9, 12.3]

print(f"Mean: {stats.mean(measurements):.2f}")
print(f"sqrt(81): {sqrt(81)}")
print(f"pi: {pi:.5f}")

### Avoid wildcard imports

```python
from math import *
```

A wildcard import places many names into the current namespace. It becomes difficult to see where names originated, and imported names can accidentally overwrite existing ones. Prefer explicit imports.

## 4. Inspecting a module

Modules are themselves Python objects. We can inspect their names, documentation, and source location.

In [ ]:
import math

print("Module name:", math.__name__)
print("A few available names:", [name for name in dir(math) if not name.startswith("_")][:10])
print("sqrt documentation:", math.sqrt.__doc__)

## 5. Creating a custom module

A custom module is created by saving Python code in a file ending in `.py`. The next cell creates `calculations.py` in the notebook's working directory.

The intended project structure is:

```text
project_folder/
├── calculations.py
└── teaching_notebook.ipynb
```

> **Teaching note:** In a normal editor, students would create `calculations.py` manually. The notebook writes the file so the complete demonstration remains self-contained.

In [ ]:
from pathlib import Path

module_code = '''"""Useful engineering calculations."""

import math

STANDARD_GRAVITY = 9.80665


def circle_area(radius):
    """Return the area of a circle."""
    return math.pi * radius**2


def celsius_to_fahrenheit(celsius):
    """Convert a Celsius temperature to Fahrenheit."""
    return celsius * 9 / 5 + 32


if __name__ == "__main__":
    print("Testing calculations.py")
    print("Area for radius 2:", circle_area(2))
'''

Path("calculations.py").write_text(module_code, encoding="utf-8")
print(Path("calculations.py").read_text(encoding="utf-8"))

## 6. Importing the custom module

Once `calculations.py` exists in the working directory, it can be imported by its filename without the `.py` extension.

In [ ]:
import calculations

area = calculations.circle_area(2.5)
temperature = calculations.celsius_to_fahrenheit(20)

print(f"Area: {area:.2f} m²")
print(f"Temperature: {temperature:.1f} °F")
print(f"Standard gravity: {calculations.STANDARD_GRAVITY} m/s²")

### What happens during an import?

In simplified form, Python:

1. searches for a module called `calculations`;
2. creates a module object;
3. executes the module's top-level statements;
4. stores its functions, constants, and other names in the module namespace;
5. binds that module object to the name `calculations` in the importing program.

Python normally caches imported modules in `sys.modules`, so a module is not repeatedly executed every time it is imported during the same session.

In [ ]:
import sys

print("Module object:", calculations)
print("Module name:", calculations.__name__)
print("Module file:", calculations.__file__)
print("Cached in sys.modules:", "calculations" in sys.modules)

## 7. The `__name__` check

Every module has a `__name__` variable.

- When a file is **imported**, `__name__` is its module name, such as `"calculations"`.
- When a file is **run directly**, `__name__` is `"__main__"`.

This pattern allows demonstration or testing code to run only when the module is executed directly:

```python
if __name__ == "__main__":
    print("Run tests or a demonstration here")
```

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "calculations.py"],
    capture_output=True,
    text=True,
    check=True,
)

print("Output when calculations.py is run directly:")
print(result.stdout)
print("Name when imported:", calculations.__name__)

## 8. How Python finds modules

Python searches directories listed in `sys.path`. These commonly include the program's directory, standard-library locations, and the environment's installed-package directory.

In [ ]:
import sys

print("First five module search locations:")
for location in sys.path[:5]:
    print(" -", location or "<current working directory>")

### A common error

```text
ModuleNotFoundError: No module named 'calculations'
```

Possible causes include:

- the filename is misspelled;
- the file is not in a searched directory;
- the wrong virtual environment is active;
- a third-party package has not been installed;
- a local file shadows another module, such as naming a file `math.py`.

## 9. Reloading a module during development

Because imported modules are cached, editing `calculations.py` does not automatically update the existing module object in a running notebook. During development, `importlib.reload()` can reload it.

> **Caution:** Reloading can become confusing when other objects already refer to names from the old module. Restarting the notebook kernel is often the cleanest option.

In [ ]:
import importlib

calculations = importlib.reload(calculations)
print("Reloaded:", calculations.__name__)

## 10. Packages: organising several modules

A **package** is a directory that groups related modules. A conventional package contains an `__init__.py` file.

The next cell creates this structure:

```text
engineering/
├── __init__.py
├── geometry.py
└── temperature.py
```

In [ ]:
from pathlib import Path

package = Path("engineering")
package.mkdir(exist_ok=True)

(package / "__init__.py").write_text(
    '"""Small teaching package for engineering calculations."""\n',
    encoding="utf-8",
)

(package / "geometry.py").write_text(
    "import math\n\ndef circle_area(radius):\n    return math.pi * radius**2\n",
    encoding="utf-8",
)

(package / "temperature.py").write_text(
    "def celsius_to_fahrenheit(value):\n    return value * 9 / 5 + 32\n",
    encoding="utf-8",
)

for path in sorted(package.iterdir()):
    print(path)

In [ ]:
from engineering.geometry import circle_area
from engineering.temperature import celsius_to_fahrenheit

print(f"Package area calculation: {circle_area(3):.2f}")
print(f"Package temperature conversion: {celsius_to_fahrenheit(25):.1f} °F")

## 11. Third-party packages

Third-party code is distributed separately from Python. It is normally installed into the active environment with `pip` and then imported in a program.

```powershell
python -m pip install bleak
```

```python
from bleak import BleakScanner
```

Installation and importing are separate operations:

```text
pip installation  →  places a distribution in the environment
import statement   →  loads a module or package into this program
```

> **Terminology note:** The name installed from PyPI and the name imported in Python are often the same, but they do not have to be.

In [ ]:
import importlib.util

module_name = "bleak"
available = importlib.util.find_spec(module_name) is not None
print(f"Is {module_name!r} available in this environment? {available}")

## 12. Check for understanding

1. What is the difference between a module and a function?
2. Why might `import math` be clearer than `from math import *`?
3. What filename would create a module named `sensors`?
4. What does Python do the first time it imports a module?
5. When does the code inside an `if __name__ == "__main__"` block run?
6. What is the difference between a module and a package?
7. Why might a notebook need to reload a module after editing its file?

<details>
<summary><strong>Suggested answers</strong></summary>

1. A function is one callable block of code; a module is a namespace/file that can contain several functions, classes, and other names.  
2. It preserves the namespace and makes each imported name's origin clear.  
3. `sensors.py`.  
4. It finds and executes the file, constructs a module object, and caches it.  
5. When that file is executed directly, not when it is imported normally.  
6. A module is usually one `.py` file; a package is a directory grouping related modules.  
7. Imports are cached, so editing the source file does not automatically replace the loaded module object.

</details>

## 13. Student exercise

Create a module called `sensors.py` containing:

- a constant called `DEFAULT_UNIT`;
- a function called `average_temperature(readings)`;
- a class called `TemperatureSensor` with `sensor_id` and `temperature` attributes.

Then import the module and:

1. create a sensor object;
2. calculate the average of three readings;
3. print the sensor and average using the default unit.

In [ ]:
# Write your solution here.


<details>
<summary><strong>One possible solution</strong></summary>

In `sensors.py`:

```python
DEFAULT_UNIT = "°C"

def average_temperature(readings):
    return sum(readings) / len(readings)

class TemperatureSensor:
    def __init__(self, sensor_id, temperature=0.0):
        self.sensor_id = sensor_id
        self.temperature = temperature
```

In the program or notebook:

```python
import sensors

sensor = sensors.TemperatureSensor("LAB-01", 21.5)
average = sensors.average_temperature([20.0, 21.5, 22.0])

print(sensor.sensor_id, sensor.temperature, sensors.DEFAULT_UNIT)
print(average, sensors.DEFAULT_UNIT)
```

</details>

## 14. Summary

- A module is usually a reusable `.py` file.
- `import` makes a module object available to a program.
- Module namespaces make the source of names clear.
- A custom module is created by saving reusable code in a `.py` file.
- `if __name__ == "__main__"` separates direct execution from imported behaviour.
- Python searches the locations in `sys.path` and caches imports in `sys.modules`.
- A package is a directory that organises related modules.
- Third-party distributions are installed into an environment before their modules can be imported.

### Exit question

Which parts of one of your existing programs would benefit from being moved into a reusable module, and what would you name that module?